In [2]:
import pandas as pd
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import calibration_and_holdout_data

clv_df = pd.read_csv("../data/processed/clv_modeling_dataset.csv")
clv_df.head()


,CustomerID,frequency,recency,T,monetary_value
0,12346.0,1,0,326,77183.600000
1,12347.0,7,365,367,23.681319
2,12348.0,4,282,358,57.975484
3,12349.0,1,0,19,24.076027
4,12350.0,1,0,310,19.670588


#### FIT BG/NBD MODEL
> Why BG/NBD?

It models:

How often customers buy

Whether they’re still “alive”

In [6]:
bgf_data = clv_df[
    ~((clv_df['frequency'] == 1) & (clv_df['recency'] == 0))
]


In [8]:
bgf = BetaGeoFitter(penalizer_coef=0.1)
bgf.fit(
    bgf_data['frequency'],
    bgf_data['recency'],
    bgf_data['T']
)


<lifetimes.BetaGeoFitter: fitted with 2845 subjects, a: 0.02, alpha: 36.90, b: 0.41, r: 1.03>

In [9]:
clv_df['predicted_purchases_6m'] = bgf.predict(
    180,
    clv_df['frequency'],
    clv_df['recency'],
    clv_df['T']
)


In [10]:
bgf.params_


r         1.032704
alpha    36.900473
a         0.019212
b         0.411977
dtype: float64

In [11]:
clv_df['predicted_purchases_6m'].describe()


count    4.338000e+03
mean     3.490892e+00
std      3.856633e+00
min      1.454114e-32
25%      1.373952e+00
50%      2.711451e+00
75%      4.578057e+00
max      9.164190e+01
Name: predicted_purchases_6m, dtype: float64

In [13]:
from lifetimes import GammaGammaFitter

# Gamma-Gamma requires frequency > 0 and monetary_value > 0
gg_data = clv_df[
    (clv_df['frequency'] > 0) &
    (clv_df['monetary_value'] > 0)
]

ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    gg_data['frequency'],
    gg_data['monetary_value']
)


<lifetimes.GammaGammaFitter: fitted with 4338 subjects, p: 3.49, q: 1.01, v: 3.18>

In [14]:
clv_df['expected_monetary_value'] = ggf.conditional_expected_average_profit(
    clv_df['frequency'],
    clv_df['monetary_value']
)


In [15]:
clv_df['clv_6m'] = ggf.customer_lifetime_value(
    bgf,
    clv_df['frequency'],
    clv_df['recency'],
    clv_df['T'],
    clv_df['monetary_value'],
    time=6,          # months
    discount_rate=0.01
)


In [16]:
clv_bi_df = clv_df[[
    'CustomerID',
    'frequency',
    'recency',
    'T',
    'monetary_value',
    'predicted_purchases_6m',
    'expected_monetary_value',
    'clv_6m'
]].copy()

clv_bi_df['CLV Segment'] = pd.qcut(
    clv_bi_df['clv_6m'],
    q=3,
    labels=['Low Value', 'Medium Value', 'High Value']
)

clv_bi_df.to_csv(
    "../data/processed/clv_scoring_dataset.csv",
    index=False
)
